# Enveda CASMI26 — Stein-Scott Fragment + Neutral-Loss Sparse Cosine Retrieval Hybrid

**Fast End-to-End Competitive Baseline:**
1. **Sparse Dual Spectral Fingerprints (`0.05 Da` bins, ~40K dims):**
   - **Fragment ions** weighted via classic Stein-Scott MS scaling $w = I^{0.5} \cdot (m/z)^{1.25}$ + L2 unit normalization.
   - **Neutral losses** ($\Delta m = \text{precursor\_mz} - m/z$) weighted via $I^{0.5}$ + L2 unit normalization to match conserved functional-group losses across structural analogs.
2. **$O(\log N)$ Precursor Mass Lookup with Multi-Tier Tolerance (`0.015 -> 0.05 -> 0.25 -> 2.0 Da`):** Ranks library hits by combined fragment + neutral-loss cosine similarity and precursor mass agreement.
3. **Multi-Spectrum Replicate Consensus & `inchikey14` Deduplication:** Aggregates candidates across replicate test spectra per `molecule_id`, deduplicates strictly by 14-char InChIKey block (`top 25` `;`-separated SMILES, `'CCO'` safe fallback).
4. **Optional De Novo Transformer Blend Hook:** Reads `/kaggle/working/denovo_predictions.csv` (if present from a trained seq2seq model) and fuses candidate ranks via Reciprocal Rank Fusion (`RRF`).


In [ ]:
import os
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

COMP_CANDIDATES = [
    Path("/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"),
    Path("/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"),
]
COMP_DIR = next(p for p in COMP_CANDIDATES if (p / "sample_submission.csv").exists())
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

try:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import rdFingerprintGenerator
    import rdkit.rdBase as rkrb
    import rdkit.RDLogger as rkl
    rkl.logger().setLevel(rkl.ERROR)
    rkrb.DisableLog("rdApp.error")
    HAS_RDKIT = True
    FP_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2)
except ImportError:
    HAS_RDKIT = False
    FP_GEN = None
    print("RDKit not pre-installed; using precomputed train.parquet inchikey14 & normalized_smiles cleanly.")

print(f"Data dir: {COMP_DIR} | RDKit available: {HAS_RDKIT}")
print("Files:", sorted(p.name for p in COMP_DIR.iterdir()))


In [ ]:
NEEDED_COLS = [
    "ingest_lib",
    "inchikey14",
    "normalized_smiles",
    "precursor_mz",
    "ms2_mzs",
    "ms2_normalized_intensities",
]

ref_df = pd.read_parquet(
    COMP_DIR / "train.parquet",
    columns=NEEDED_COLS,
    filters=[("ingest_lib", "!=", "enveda-180")],
)
# Drop null masses/structures & sort strictly by precursor_mz for O(log N) binary window slicing
ref_df = ref_df.dropna(subset=["precursor_mz", "inchikey14", "normalized_smiles"])
ref_df = ref_df.sort_values("precursor_mz", kind="mergesort").reset_index(drop=True)
print(f"Reference spectra: {len(ref_df):,} | Unique inchikey14 structures: {ref_df['inchikey14'].nunique():,}")


In [ ]:
BIN_RES = 20.0          # 0.05 Da bins
MAX_MZ_BIN = 40000      # 0..2000 Da
MAX_PEAKS = 128

def encode_spectra_csr(
    pmzs: np.ndarray,
    mzs_list,
    ints_list,
    bin_res: float = BIN_RES,
    max_bin: int = MAX_MZ_BIN,
    max_peaks: int = MAX_PEAKS,
) -> tuple[sparse.csr_matrix, sparse.csr_matrix]:
    '''Builds L2-normalized CSR matrices for Stein-Scott fragment m/z & neutral-loss vectors.'''
    n_rows = len(pmzs)
    f_indptr = np.zeros(n_rows + 1, dtype=np.int32)
    f_indices = []
    f_data = []

    nl_indptr = np.zeros(n_rows + 1, dtype=np.int32)
    nl_indices = []
    nl_data = []

    f_cnt = 0
    nl_cnt = 0
    for i in range(n_rows):
        pmz = float(pmzs[i])
        raw_mzs = mzs_list[i]
        raw_ints = ints_list[i]
        if raw_mzs is None or raw_ints is None or not np.isfinite(pmz):
            f_indptr[i + 1] = f_cnt
            nl_indptr[i + 1] = nl_cnt
            continue
        mzs = np.atleast_1d(np.asarray(raw_mzs, dtype=np.float32))
        ints = np.atleast_1d(np.asarray(raw_ints, dtype=np.float32))
        fin = np.isfinite(mzs) & np.isfinite(ints)
        if not np.all(fin):
            mzs = mzs[fin]
            ints = ints[fin]
        if mzs.size == 0 or ints.size == 0:
            f_indptr[i + 1] = f_cnt
            nl_indptr[i + 1] = nl_cnt
            continue

        if len(mzs) > max_peaks:
            top_idx = np.argpartition(ints, -max_peaks)[-max_peaks:]
            mzs = mzs[top_idx]
            ints = ints[top_idx]

        max_int = float(ints.max())
        if max_int > 0.0:
            ints = ints / max_int
        ints = np.clip(ints, 0.0, 1.0)

        # Fragment vector (Stein-Scott mass-weighted sqrt intensity)
        f_bins = np.rint(mzs * bin_res).astype(np.int32)
        valid_f = (f_bins >= 0) & (f_bins < max_bin) & (ints > 0.0)
        if np.any(valid_f):
            fb = f_bins[valid_f]
            fw = np.sqrt(ints[valid_f]) * np.power(np.maximum(mzs[valid_f], 1.0), 1.25)
            # Merge duplicate bins via bincount or sorted summation
            order = np.argsort(fb)
            fb_s = fb[order]
            fw_s = fw[order]
            uniq_mask = np.concatenate(([True], fb_s[1:] != fb_s[:-1]))
            u_bins = fb_s[uniq_mask]
            u_weights = np.add.reduceat(fw_s, np.flatnonzero(uniq_mask)).astype(np.float32)
            norm = float(np.linalg.norm(u_weights))
            if norm > 0.0:
                u_weights /= norm
            f_indices.append(u_bins)
            f_data.append(u_weights)
            f_cnt += len(u_bins)
        f_indptr[i + 1] = f_cnt

        # Neutral-loss vector (precursor_mz - fragment_mz)
        nl_vals = pmz - mzs
        nl_bins = np.rint(nl_vals * bin_res).astype(np.int32)
        valid_nl = (nl_bins >= 1) & (nl_bins < max_bin) & (ints > 0.0)
        if np.any(valid_nl):
            nb = nl_bins[valid_nl]
            nw = np.sqrt(ints[valid_nl])
            order = np.argsort(nb)
            nb_s = nb[order]
            nw_s = nw[order]
            uniq_mask = np.concatenate(([True], nb_s[1:] != nb_s[:-1]))
            u_bins = nb_s[uniq_mask]
            u_weights = np.add.reduceat(nw_s, np.flatnonzero(uniq_mask)).astype(np.float32)
            norm = float(np.linalg.norm(u_weights))
            if norm > 0.0:
                u_weights /= norm
            nl_indices.append(u_bins)
            nl_data.append(u_weights)
            nl_cnt += len(u_bins)
        nl_indptr[i + 1] = nl_cnt

    X_f = sparse.csr_matrix(
        (
            np.concatenate(f_data) if f_data else np.empty(0, dtype=np.float32),
            np.concatenate(f_indices) if f_indices else np.empty(0, dtype=np.int32),
            f_indptr,
        ),
        shape=(n_rows, max_bin),
        dtype=np.float32,
    )
    X_nl = sparse.csr_matrix(
        (
            np.concatenate(nl_data) if nl_data else np.empty(0, dtype=np.float32),
            np.concatenate(nl_indices) if nl_indices else np.empty(0, dtype=np.int32),
            nl_indptr,
        ),
        shape=(n_rows, max_bin),
        dtype=np.float32,
    )
    return X_f, X_nl


ref_pmz = ref_df["precursor_mz"].to_numpy(dtype=np.float64)
ref_ik = ref_df["inchikey14"].to_numpy()
ref_smiles = ref_df["normalized_smiles"].to_numpy()

X_ref_frag, X_ref_nl = encode_spectra_csr(
    ref_pmz,
    ref_df["ms2_mzs"].to_numpy(),
    ref_df["ms2_normalized_intensities"].to_numpy(),
)
print(f"Built sparse index: X_ref_frag={X_ref_frag.shape} nnz={X_ref_frag.nnz:,} | X_ref_nl nnz={X_ref_nl.nnz:,}")


In [ ]:
def retrieve_candidates(
    q_pmzs: np.ndarray,
    Q_frag: sparse.csr_matrix,
    Q_nl: sparse.csr_matrix,
    ref_pmz_arr: np.ndarray,
    ref_frag: sparse.csr_matrix,
    ref_nl: sparse.csr_matrix,
    ref_ik_arr: np.ndarray,
    ref_smiles_arr: np.ndarray,
    exclude_mask: np.ndarray | None = None,
    top_k_per_spec: int = 40,
    tolerances: tuple[float, ...] = (0.015, 0.05, 0.25, 2.0),
) -> list[list[tuple[float, str, str]]]:
    '''Retrieves ranked (score, inchikey14, smiles) candidates for each query spectrum.'''
    results = []
    n_ref = len(ref_pmz_arr)

    for i in range(len(q_pmzs)):
        pmz = float(q_pmzs[i])
        qf = Q_frag[i]
        qn = Q_nl[i]

        ik_best: dict[str, tuple[float, str]] = {}
        for tol in tolerances:
            lo = int(np.searchsorted(ref_pmz_arr, pmz - tol, side="left"))
            hi = int(np.searchsorted(ref_pmz_arr, pmz + tol, side="right"))
            if hi <= lo:
                continue

            sub_f = ref_frag[lo:hi]
            sub_nl = ref_nl[lo:hi]
            sim_f = (sub_f @ qf.T).toarray().ravel()
            sim_nl = (sub_nl @ qn.T).toarray().ravel()
            dmz = np.abs(ref_pmz_arr[lo:hi] - pmz)
            scores = 0.70 * sim_f + 0.30 * sim_nl - 0.15 * dmz + 0.08 * (dmz <= 0.015)

            if exclude_mask is not None:
                excl = exclude_mask[lo:hi]
                if np.any(excl):
                    scores = np.where(excl, -1e9, scores)

            # Take top candidates in this slice
            k_slice = min(len(scores), top_k_per_spec * 3)
            if k_slice <= 0:
                continue
            cand_idx = np.argpartition(scores, -k_slice)[-k_slice:] if len(scores) > k_slice else np.arange(len(scores))
            cand_idx = cand_idx[np.argsort(scores[cand_idx])[::-1]]

            for ridx in cand_idx:
                sc = float(scores[ridx])
                if sc < -1e8:
                    continue
                gidx = lo + int(ridx)
                ik = str(ref_ik_arr[gidx])
                smi = str(ref_smiles_arr[gidx])
                prev = ik_best.get(ik)
                if prev is None or sc > prev[0]:
                    ik_best[ik] = (sc, smi)

            if len(ik_best) >= top_k_per_spec:
                break

        ranked = sorted(((sc, ik, smi) for ik, (sc, smi) in ik_best.items()), reverse=True)[:top_k_per_spec]
        results.append(ranked)
    return results


In [ ]:
# Local validation check: replicate recovery + structure-disjoint analog Tanimoto
rng = np.random.default_rng(7)
val_n = min(300, len(ref_df))
val_indices = rng.choice(len(ref_df), size=val_n, replace=False)

# Test A: Self-excluded library lookup (mirrors replicate spectra / analog retrieval)
excl = np.zeros(len(ref_df), dtype=bool)
excl[val_indices] = True

# Build a clean per-row boolean self-exclusion check using index checks in loop for small val sample:
hit1, hit5, hit25 = 0, 0, 0
tanimotos_1 = []
for row_pos in val_indices:
    pmz = float(ref_pmz[row_pos])
    qf = X_ref_frag[row_pos]
    qn = X_ref_nl[row_pos]
    true_ik = str(ref_ik[row_pos])
    true_smi = str(ref_smiles[row_pos])

    lo = int(np.searchsorted(ref_pmz, pmz - 0.05, side="left"))
    hi = int(np.searchsorted(ref_pmz, pmz + 0.05, side="right"))
    if hi <= lo:
        continue
    sf = (X_ref_frag[lo:hi] @ qf.T).toarray().ravel()
    snl = (X_ref_nl[lo:hi] @ qn.T).toarray().ravel()
    dmz = np.abs(ref_pmz[lo:hi] - pmz)
    scores = 0.70 * sf + 0.30 * snl - 0.15 * dmz
    if lo <= row_pos < hi:
        scores[row_pos - lo] = -1e9  # Exclude the exact query spectrum itself

    top_iks = []
    seen = set()
    order = np.argsort(scores)[::-1]
    top_smi = None
    for j in order:
        if scores[j] < -1e8:
            continue
        ik = str(ref_ik[lo + int(j)])
        if ik not in seen:
            seen.add(ik)
            top_iks.append(ik)
            if top_smi is None:
                top_smi = str(ref_smiles[lo + int(j)])
            if len(top_iks) >= 25:
                break

    if top_iks:
        hit1 += int(top_iks[0] == true_ik)
        hit5 += int(true_ik in top_iks[:5])
        hit25 += int(true_ik in top_iks[:25])
        if HAS_RDKIT and top_smi is not None:
            m1, m2 = Chem.MolFromSmiles(top_smi), Chem.MolFromSmiles(true_smi)
            if m1 is not None and m2 is not None:
                tanimotos_1.append(DataStructs.TanimotoSimilarity(FP_GEN.GetSparseFingerprint(m1), FP_GEN.GetSparseFingerprint(m2)))

print(f"Replicate-Spectrum Retrieval Validation (n={val_n}): Top-1={hit1/val_n:.4f} | Top-5={hit5/val_n:.4f} | Top-25={hit25/val_n:.4f}")
if tanimotos_1:
    print(f"Mean Top-1 Morgan r=2 Tanimoto: {float(np.mean(tanimotos_1)):.4f}")


In [ ]:
# Predict on test.parquet, pool replicate spectra per molecule_id, optional De Novo RRF blend, and emit submission.csv
test_df = pd.read_parquet(COMP_DIR / "test.parquet")
print(f"Test rows: {len(test_df):,} spectra | Unique molecule_ids: {test_df['molecule_id'].nunique():,}")

X_te_frag, X_te_nl = encode_spectra_csr(
    test_df["precursor_mz"].to_numpy(dtype=np.float64),
    test_df["ms2_mzs"].to_numpy(),
    test_df["ms2_normalized_intensities"].to_numpy(),
)
per_spec_hits = retrieve_candidates(
    test_df["precursor_mz"].to_numpy(dtype=np.float64),
    X_te_frag,
    X_te_nl,
    ref_pmz,
    X_ref_frag,
    X_ref_nl,
    ref_ik,
    ref_smiles,
    top_k_per_spec=40,
)

N_GUESSES = 25
FALLBACK_SMILES = "CCO"

# Multi-spectrum pooling: combine max similarity + replicate consistency bonus
mol_ik_scores: dict[str, dict[str, list[tuple[float, str]]]] = defaultdict(lambda: defaultdict(list))
for mol_id, hits in zip(test_df["molecule_id"].tolist(), per_spec_hits):
    for sc, ik, smi in hits:
        mol_ik_scores[mol_id][ik].append((sc, smi))

# Optional De Novo Transformer blend hook (save molecule_id,smiles from a generative model to fuse ranks)
DENOVO_PATH = WORKING_DIR / "denovo_predictions.csv"
denovo_map: dict[str, list[str]] = {}
if DENOVO_PATH.exists():
    dn_df = pd.read_csv(DENOVO_PATH)
    for row in dn_df.itertuples(index=False):
        denovo_map[str(row.molecule_id)] = [s for s in str(row.smiles).split(";") if s]
    print(f"Fusing {len(denovo_map):,} De Novo prediction lists via RRF.")


def build_molecule_guesses(mol_id: str) -> str:
    ik_dict = mol_ik_scores.get(mol_id, {})
    pooled: list[tuple[float, str, str]] = []
    for ik, occs in ik_dict.items():
        occs_sorted = sorted(occs, reverse=True)
        agg = occs_sorted[0][0] + 0.08 * (len(occs_sorted) - 1)
        pooled.append((agg, ik, occs_sorted[0][1]))
    pooled.sort(reverse=True)

    # Blend RRF scores if De Novo candidates exist
    rrf: dict[str, float] = {}
    smiles_for_ik: dict[str, str] = {}
    for rank, (_, ik, smi) in enumerate(pooled[:40]):
        rrf[ik] = rrf.get(ik, 0.0) + 1.0 / (12.0 + rank)
        smiles_for_ik[ik] = smi

    for rank, smi in enumerate(denovo_map.get(mol_id, [])[:25]):
        if HAS_RDKIT:
            m = Chem.MolFromSmiles(smi)
            if m is None:
                continue
            ik = Chem.MolToInchiKey(m).split("-")[0]
            can_smi = Chem.MolToSmiles(m)
        else:
            ik = smi
            can_smi = smi
        rrf[ik] = rrf.get(ik, 0.0) + 0.65 / (12.0 + rank)
        smiles_for_ik.setdefault(ik, can_smi)

    ranked_iks = sorted(rrf.items(), key=lambda kv: kv[1], reverse=True)
    guesses = [smiles_for_ik[ik] for ik, _ in ranked_iks[:N_GUESSES]]
    return ";".join(guesses) if guesses else FALLBACK_SMILES


sub = pd.read_csv(COMP_DIR / "sample_submission.csv")
sub["smiles"] = sub["molecule_id"].astype(str).apply(build_molecule_guesses)
out_csv = WORKING_DIR / "submission.csv"
sub.to_csv(out_csv, index=False)

n_guess = sub["smiles"].str.split(";").map(len)
assert len(sub) > 0 and sub["smiles"].notna().all() and (n_guess >= 1).all() and (n_guess <= 25).all()
print(f"Saved {out_csv} with {len(sub):,} rows | Guesses median={int(n_guess.median())}, max={int(n_guess.max())}")
sub.head(10)
